# Taichi Splatting Type Mismatch - Reproduction & Fix Proof

This notebook demonstrates the `BeartypeCallHintParamViolation` error and verifies that the proposed fix (wrapping params) works correctly. 

We use **mock classes** to simulate the behavior of the `taichi_splatting` library, so you can run this without needing the full library installed.

In [ ]:
import torch
from dataclasses import dataclass

# --- MOCKING THE LIBRARY STRUCTURE ---

# 1. Simulate Gaussians3D (The type render_gaussians expects)
# In the real library, this is a dataclass holding the tensors.
@dataclass
class Gaussians3D:
    position: torch.Tensor
    log_scaling: torch.Tensor
    rotation: torch.Tensor
    alpha_logit: torch.Tensor
    feature: torch.Tensor

# 2. Simulate ParameterClass (The type used in our training loop)
# This class manages optimization but IS NOT a Gaussians3D instance.
class ParameterClass:
    def __init__(self):
        # Mock Data (using small tensors)
        self.position = torch.tensor([0.0])
        self.log_scaling = torch.tensor([0.0])
        self.rotation = torch.tensor([0.0])
        self.alpha_logit = torch.tensor([0.0])
        self.feature = torch.tensor([0.0])

# 3. Simulate the render_gaussians function with strict type checking
# The real library uses @beartype which checks arguments at runtime.
def render_gaussians(gaussians, cam=None, config=None, compute_split_heuristics=False):
    # Simulating the strict check that caused the error
    if not isinstance(gaussians, Gaussians3D):
        raise TypeError(
            f"BeartypeCallHintParamViolation: Function render_gaussians() parameter gaussians="
            f"{gaussians} violates type hint <class 'taichi_splatting.data_types.Gaussians3D'>, "
            f"as {type(gaussians)} is not instance of <class 'taichi_splatting.data_types.Gaussians3D'>."
        )
    
    # If check passes:
    print("✅ Success! render_gaussians accepted the input object.")
    return "Rendering Output Image"

## Test 1: Reproducing the Error
This cell runs the code **as it was before the fix**. We pass the `ParameterClass` directly to the renderer.
**Expected Outcome:** It should fail with a `TypeError`.

In [ ]:
params = ParameterClass()

print("Attempting to call render_gaussians(params)... (Original Code)")
try:
    render_gaussians(params)
except TypeError as e:
    print(f"\n❌ Caught Expected Error:\n{e}")

## Test 2: Verifying the Fix
This cell runs the code **with the fix**. We wrap the `params` data into a fresh `Gaussians3D` object.
**Expected Outcome:** It should succeed.

In [ ]:
params = ParameterClass()

print("Applying Fix: Wrapping params in Gaussians3D...")

# --- THE FIX ---
gaussians_wrapper = Gaussians3D(
    position=params.position,
    log_scaling=params.log_scaling,
    rotation=params.rotation,
    alpha_logit=params.alpha_logit,
    feature=params.feature
)
# ----------------

render_gaussians(gaussians_wrapper)